# Name That LLM · 第二轮优化

字符 TF-IDF、格式与文风特征，比较不同正则强度的线性 SVM 及其与 Extra Trees 的组合。
本地默认执行嵌套分组验证。Kaggle 默认用已选配置在完整训练集拟合，输出 `submission.parquet`。
本地可用环境变量 `NAME_THAT_LLM_DATA` 指定数据目录，设 `NAME_THAT_LLM_VALIDATE=0` 可直接运行提交配置。
在 Kaggle 导入 Notebook、挂载竞赛数据、关闭互联网，保存运行版本后提交。


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score, recall_score
from sklearn.model_selection import GroupKFold
from sklearn.svm import LinearSVC

KAGGLE_DATA = Path('/kaggle/input/competitions/name-that-llm')
LOCAL_DATA = Path('data')
if not LOCAL_DATA.exists():
    LOCAL_DATA = Path.cwd().parent / 'data'
DATA_DIR = Path(os.environ['NAME_THAT_LLM_DATA']) if 'NAME_THAT_LLM_DATA' in os.environ else (
    KAGGLE_DATA if (KAGGLE_DATA / 'train.csv').exists() else LOCAL_DATA
)
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
N_SPLITS = 5
print(f'Data: {DATA_DIR.resolve()}')
print(f'Output: {OUTPUT_DIR.resolve()}')
VALIDATE = os.environ.get("NAME_THAT_LLM_VALIDATE", "0" if Path("/kaggle/working").is_dir() else "1") == "1"

FROZEN_CANDIDATE = 'blend_c1_w25'

In [ ]:
train_path = DATA_DIR / 'train.csv'
test_path = DATA_DIR / 'test.csv'
if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(f'Expected train.csv and test.csv in {DATA_DIR}')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
required_train = {'id', 'prompt_id', 'text', 'model'}
required_test = {'id', 'text'}
if not required_train.issubset(train.columns):
    raise ValueError(f'Train columns missing: {sorted(required_train - set(train.columns))}')
if not required_test.issubset(test.columns):
    raise ValueError(f'Test columns missing: {sorted(required_test - set(test.columns))}')
if train[list(required_train)].isna().any().any() or test[list(required_test)].isna().any().any():
    raise ValueError('Required columns contain null values')
if train['id'].duplicated().any() or test['id'].duplicated().any():
    raise ValueError('IDs must be unique within each file')
if (train['text'].astype(str).str.strip() == '').any() or (test['text'].astype(str).str.strip() == '').any():
    raise ValueError('Text cannot be empty')

labels = sorted(train['model'].unique().tolist())
if len(labels) != 18:
    raise ValueError(f'Expected 18 model labels, found {len(labels)}')
if train['prompt_id'].nunique() < N_SPLITS:
    raise ValueError(f'Need at least {N_SPLITS} distinct prompt_id values')

print(f'Train rows: {len(train):,}; test rows: {len(test):,}')
print(f'Prompts: {train["prompt_id"].nunique()}; classes: {len(labels)}')
print(train['model'].value_counts().sort_index().to_string())

In [ ]:
import re
from collections import Counter
from scipy.special import softmax
from scipy.sparse import hstack
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from threadpoolctl import threadpool_limits
from sklearn.preprocessing import StandardScaler

STYLE_PATTERNS = (
    r'[,;:]', r'[?!]', r'[—–]', r'-', r'…|\.\.\.', r'[’‘“”]',
    r'''['"]''', r'[()]', r'[\[\]{}]', r'\*\*', r'`', r'\n',
    r'(?m)^\s*(?:[-*] |\d+\. )', r'(?m)^#{1,6} ', r'\d', r'[A-Z]',
    r'(?i)\b(?:i|me|my|we|our|you|your)\b',
    r'(?i)\b(?:however|therefore|indeed|thus|rather|simply)\b',
)
STYLE_PATTERNS = tuple(re.compile(pattern) for pattern in STYLE_PATTERNS)
WORD = re.compile(r"\b[\w’']+\b")
SENTENCE_BREAK = re.compile(r"[.!?]+(?:\s|$)")

def style_features(texts):
    rows = []
    for text in texts:
        tokens = WORD.findall(text.lower())
        word_count = max(len(tokens), 1)
        sentences = [part for part in SENTENCE_BREAK.split(text) if part.strip()]
        sentence_lengths = [len(WORD.findall(part)) for part in sentences] or [0]
        lengths = [len(token) for token in tokens] or [0]
        statistics = [
            np.log1p(len(text)), np.log1p(word_count),
            np.mean(lengths), np.std(lengths), np.mean(sentence_lengths),
            np.std(sentence_lengths), len(set(tokens)) / word_count,
        ]
        rates = [100 * len(pattern.findall(text)) / word_count for pattern in STYLE_PATTERNS]
        rows.append(statistics + rates)
    return np.asarray(rows, dtype=np.float32)


FUNCTION_WORDS = (
    "a an the this that those these it its there one on in at over under to of for with from by as into "
    "is are was be been being can could may might must shall should will would or and but if while so "
    "because although then than also even just still often only very more most some any such which who "
    "what how when all each both no not like between through about across within without we you i"
).split()
MARKS = tuple(".,;:!?()[]{}") + ("—", "–", "-", "'", "’", '"', "“", "”", "*", "`", "_", "/", "\n")

def rich_style(texts):
    base = style_features(texts)
    rows = []
    for text in texts:
        tokens = WORD.findall(text.lower())
        n = max(len(tokens), 1)
        counts = Counter(tokens)
        sentences = [part for part in SENTENCE_BREAK.split(text) if part.strip()]
        lengths = np.asarray([len(WORD.findall(part)) for part in sentences] or [0])
        rows.append(
            [100 * counts[word] / n for word in FUNCTION_WORDS]
            + [100 * text.count(mark) / n for mark in MARKS]
            + [float(mark in text) for mark in MARKS]
            + [float(np.median(lengths)), float(lengths[0]), float(lengths[-1]),
               sum(len(token) >= 8 for token in tokens) / n,
               sum(len(token) <= 3 for token in tokens) / n,
               sum(count == 1 for count in counts.values()) / n]
        )
    return np.hstack((base, np.asarray(rows, dtype=np.float32)))


def shape(text):
    # Keep whitespace and punctuation while replacing the individual letters/digits.
    return re.sub(r"[^\W\d_]", lambda match: "A" if match[0].isupper() else "a",
                  re.sub(r"\d", "0", text))


def make_tfidf(fit_text, other_text, *, shape_only=False):
    if shape_only:
        fit_text, other_text = fit_text.map(shape), other_text.map(shape)
    vectorizer = TfidfVectorizer(
        analyzer="char", ngram_range=(2, 5) if not shape_only else (2, 6),
        max_features=80_000 if not shape_only else 20_000,
        min_df=2, sublinear_tf=True, dtype=np.float32, lowercase=not shape_only,
    )
    return vectorizer.fit_transform(fit_text), vectorizer.transform(other_text)


def scale_style(fit_style, other_style):
    scaler = StandardScaler()
    return np.clip(scaler.fit_transform(fit_style), -5, 5), np.clip(scaler.transform(other_style), -5, 5)


def prepare(fit, other, requested):
    train_text, other_text = fit["text"].astype(str), other["text"].astype(str)
    char_fit, char_other = make_tfidf(train_text, other_text)
    old_fit, old_other = scale_style(style_features(train_text), style_features(other_text))
    feature_fit = {"base": hstack((char_fit, 0.10 * old_fit), format="csr")}
    feature_other = {"base": hstack((char_other, 0.10 * old_other), format="csr")}
    needed = {CANDIDATES[name][0] for name in requested}
    rich_fit, rich_other = None, None
    if "rich" in needed or any(CANDIDATES[name][2] > 0 for name in requested):
        rich_fit, rich_other = rich_style(train_text), rich_style(other_text)
    if "rich" in needed:
        shape_fit, shape_other = make_tfidf(train_text, other_text, shape_only=True)
        fit_scaled, other_scaled = scale_style(rich_fit, rich_other)
        feature_fit["rich"] = hstack((char_fit, 0.65 * shape_fit, 0.075 * fit_scaled), format="csr")
        feature_other["rich"] = hstack((char_other, 0.65 * shape_other, 0.075 * other_scaled), format="csr")
    return feature_fit, feature_other, rich_fit, rich_other


# Values are (feature set, SVM C, nonlinear model weight). Fixed before running CV.
BASELINE = "round1_baseline"
CANDIDATES = {
    BASELINE: ("base", 1.0, 0.0),
    "baseline_c005": ("base", 0.05, 0.0),
    "baseline_regularized": ("base", 0.25, 0.0),
    "rich_c005": ("rich", 0.05, 0.0),
    "rich_c025": ("rich", 0.25, 0.0),
    "rich_c1": ("rich", 1.0, 0.0),
    "rich_c4": ("rich", 4.0, 0.0),
    "blend_c025_w25": ("rich", 0.25, 0.25),
    "blend_c1_w25": ("rich", 1.0, 0.25),
    "blend_c1_w50": ("rich", 1.0, 0.50),
    "blend_c005_w50": ("rich", 0.05, 0.50),
    "blend_c025_w75": ("rich", 0.25, 0.75),
    "blend_c4_w25": ("rich", 4.0, 0.25),
    "forest_style_only": ("style", None, 1.0),
}


def predict_candidates(fit, other, requested, measure_train=False):
    x_fit, x_other, rich_fit, rich_other = prepare(fit, other, requested)
    y_fit = fit["model"]
    assert sorted(y_fit.unique()) == labels
    forest_prob, forest_train_prob = None, None
    if any(CANDIDATES[name][2] > 0 for name in requested):
        forest = ExtraTreesClassifier(
            n_estimators=240, min_samples_leaf=4, max_features=0.65,
            n_jobs=4, random_state=42, class_weight="balanced",
        )
        forest.fit(rich_fit, y_fit)
        forest_prob = forest.predict_proba(rich_other)
        if measure_train:
            forest_train_prob = forest.predict_proba(rich_fit)

    predictions, training_predictions, linear_cache = {}, {}, {}
    for name in requested:
        features, c, weight = CANDIDATES[name]
        if weight == 1.0:
            predictions[name] = np.asarray(labels)[forest_prob.argmax(axis=1)]
            if measure_train:
                training_predictions[name] = np.asarray(labels)[forest_train_prob.argmax(axis=1)]
            continue
        if (features, c) not in linear_cache:
            svm = LinearSVC(C=c, class_weight="balanced", max_iter=10_000, random_state=42)
            svm.fit(x_fit[features], y_fit)
            margins = svm.decision_function(x_other[features])
            train_margins = svm.decision_function(x_fit[features]) if measure_train else None
            linear_cache[features, c] = margins, train_margins
        margins, train_margins = linear_cache[features, c]
        scores = (1 - weight) * softmax(3 * margins, axis=1) + weight * forest_prob if weight else margins
        predictions[name] = np.asarray(labels)[scores.argmax(axis=1)]
        if measure_train:
            train_scores = ((1 - weight) * softmax(3 * train_margins, axis=1)
                            + weight * forest_train_prob if weight else train_margins)
            training_predictions[name] = np.asarray(labels)[train_scores.argmax(axis=1)]
    return predictions, training_predictions


In [ ]:

def split_indices(frame, folds, seed):
    splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
    for fit_idx, score_idx in splitter.split(frame["text"], frame["model"], frame["prompt_id"]):
        fit, score = frame.iloc[fit_idx], frame.iloc[score_idx]
        assert set(fit["prompt_id"]).isdisjoint(score["prompt_id"])
        assert sorted(fit["model"].unique()) == sorted(score["model"].unique()) == labels
        yield fit_idx, score_idx


def tune(frame, seed):
    scores = {name: [] for name in CANDIDATES}
    gaps = {name: [] for name in CANDIDATES}
    for fit_idx, val_idx in split_indices(frame, 3, seed):
        fit, valid = frame.iloc[fit_idx], frame.iloc[val_idx]
        predictions, trained = predict_candidates(fit, valid, list(CANDIDATES), measure_train=True)
        for name, pred in predictions.items():
            score = balanced_accuracy_score(valid["model"], pred)
            scores[name].append(score)
            gaps[name].append(balanced_accuracy_score(fit["model"], trained[name]) - score)
    ranking = pd.DataFrame({
        "inner_mean": pd.Series({name: np.mean(values) for name, values in scores.items()}),
        "inner_gap": pd.Series({name: np.mean(values) for name, values in gaps.items()}),
    }).sort_values("inner_mean", ascending=False)
    # Treat candidates within 0.5 percentage points as close enough to compare gaps.
    comparable = ranking[ranking["inner_mean"] >= ranking["inner_mean"].max() - 0.005]
    selected = comparable["inner_gap"].idxmin()
    ranking["selected"] = ranking.index == selected
    return selected, ranking


if VALIDATE:
    nested_predictions = np.empty(len(train), dtype=object)
    baseline_predictions = np.empty(len(train), dtype=object)
    outer_rows = []
    fixed_rows = []
    fixed_names = ("rich_c005", "blend_c1_w25")
    with threadpool_limits(limits=4):
        for fold, (dev_idx, hold_idx) in enumerate(split_indices(train, 5, 193), start=1):
            dev, hold = train.iloc[dev_idx], train.iloc[hold_idx]
            selected, inner_scores = tune(dev, 700 + fold)
            requested = list(dict.fromkeys((BASELINE, selected, *fixed_names)))
            prediction, train_prediction = predict_candidates(dev, hold, requested, measure_train=True)
            for name in fixed_names:
                train_score = balanced_accuracy_score(dev["model"], train_prediction[name])
                valid_score = balanced_accuracy_score(hold["model"], prediction[name])
                fixed_rows.append(dict(fold=fold, candidate=name, train_score=train_score,
                                       validation=valid_score, gap=train_score - valid_score))
            nested_predictions[hold_idx] = prediction[selected]
            baseline_predictions[hold_idx] = prediction[BASELINE]
            row = {
                "fold": fold, "holdout_rows": len(hold), "selected": selected,
                "inner_score": inner_scores.at[selected, "inner_mean"],
                "selected_train": balanced_accuracy_score(dev["model"], train_prediction[selected]),
                "selected_validation": balanced_accuracy_score(hold["model"], prediction[selected]),
                "baseline_train": balanced_accuracy_score(dev["model"], train_prediction[BASELINE]),
                "baseline_validation": balanced_accuracy_score(hold["model"], prediction[BASELINE]),
            }
            outer_rows.append(row)
            print(row, flush=True)

        final_choice, final_scores = tune(train, 2026)
    folds = pd.DataFrame(outer_rows)
    folds["selected_gap"] = folds["selected_train"] - folds["selected_validation"]
    folds["baseline_gap"] = folds["baseline_train"] - folds["baseline_validation"]
    recalls = pd.DataFrame({
        "model": labels, "support": train["model"].value_counts().reindex(labels).to_numpy(),
        "nested_recall": recall_score(train["model"], nested_predictions, labels=labels, average=None),
        "baseline_recall": recall_score(train["model"], baseline_predictions, labels=labels, average=None),
    })
    recalls["change"] = recalls["nested_recall"] - recalls["baseline_recall"]
    confusion = pd.DataFrame(
        confusion_matrix(train["model"], nested_predictions, labels=labels), index=labels, columns=labels,
    )
    folds.to_csv(OUTPUT_DIR / "round2_nested_folds.csv", index=False)
    pd.DataFrame(fixed_rows).to_csv(OUTPUT_DIR / "round2_fixed_comparison.csv", index=False)
    recalls.to_csv(OUTPUT_DIR / "round2_class_recall.csv", index=False)
    confusion.to_csv(OUTPUT_DIR / "round2_confusion.csv", index_label="true_model")
    final_scores.to_csv(OUTPUT_DIR / "round2_final_tuning.csv", index_label="candidate")
    print(final_scores.to_string())
    print(f"Final full-data selection: {final_choice}", flush=True)


In [ ]:
if not VALIDATE:
    final_choice = FROZEN_CANDIDATE
with threadpool_limits(limits=4):
    predictions, _ = predict_candidates(train, test, [final_choice])
predictions = predictions[final_choice]
print(f"Prediction algorithm: {final_choice}")
submission = pd.DataFrame({'id': test['id'], 'model': predictions})

if len(submission) != len(test):
    raise ValueError('Submission row count does not match test')
if submission['id'].isna().any() or submission['model'].isna().any():
    raise ValueError('Submission contains null values')
if submission['id'].duplicated().any() or set(submission['id']) != set(test['id']):
    raise ValueError('Submission IDs do not match test IDs exactly')
if not set(submission['model']).issubset(set(labels)):
    raise ValueError('Submission contains an invalid model label')

submission_path = OUTPUT_DIR / 'submission.parquet'
submission.to_parquet(submission_path, index=False)
check = pd.read_parquet(submission_path)
if list(check.columns) != ['id', 'model'] or len(check) != len(test):
    raise ValueError('Saved parquet file failed verification')
print(f'Saved {submission_path} ({len(submission):,} rows)')
display(submission.head())